# Evaluation — Italian WITS

Full evaluation with traditional metrics, abstraction metrics,
and LLM-as-Judge scoring.

**Memory strategy**: SigExt runs on CPU → unloaded → single LLM for both summary + judge.

In [1]:
!uv pip install -e ../..

Using Python 3.12.11 environment at: /home/zeus/miniconda3/envs/cloudspace
Resolved 133 packages in 5.09s                                       
   Building sm-sip @ file:///teamspace/studios/this_studio/sm-sip      
   Building sm-sip @ file:///teamspace/studios/this_studio/sm-sip1A
   Building sm-sip @ file:///teamspace/studios/this_studio/sm-sip1A
   Building sm-sip @ file:///teamspace/studios/this_studio/sm-sip1A
   Building sm-sip @ file:///teamspace/studios/this_studio/sm-sip1A
   Building sm-sip @ file:///teamspace/studios/this_studio/sm-sip1A
      Built sm-sip @ file:///teamspace/studios/this_studio/sm-sip1A
Prepared 1 package in 1.11s                                              
Uninstalled 1 package in 2ms
Installed 1 package in 21msfile:///teamspace/studios/this_st
 ~ sm-sip==1.0.0 (from file:///teamspace/studios/this_studio/sm-sip)


In [2]:
from huggingface_hub import login
login()

## Configuration

In [3]:
from sm_sip.config import SigExtConfig, InferenceConfig, EvalConfig

sigext_config = SigExtConfig.from_preset("it", "10k-60t")
inference_config = InferenceConfig(lang="it", quantization="4bit", prompt_type="source_aware")
eval_config = EvalConfig(lang="it", judge_model_id="Qwen/Qwen2.5-14B-Instruct")

## Step 1: Load Data & Preprocess with SigExt (CPU)

In [4]:
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, unload_sigext_model, preprocess_dataset

test_data = get_test_data(lang="it", num_samples=100, skip_samples=sigext_config.skip_samples)

# SigExt runs on CPU to save GPU VRAM for the LLM
sigext_model, sigext_tokenizer = load_sigext_model(sigext_config.model_id, device="cpu")
processed_data = preprocess_dataset(test_data, sigext_model, sigext_tokenizer, lang="it")

# Free memory before loading LLM
unload_sigext_model(sigext_model, sigext_tokenizer)
print(f"Preprocessed {len(processed_data)} samples.")

  Loading IT test data (skipping 25000)...


Repo card metadata block was not found. Setting CardData to empty.


  Loaded 100 WITS samples (skipped 25000)
  Loading SigExt model on cpu: LookUpMark/sigext-wits-it-10k-060t...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Extracting salient sentences:   0%|          | 0/100 [00:00<?, ?it/s]

  SigExt model unloaded.
Preprocessed 100 samples.


## Step 2: Load Single LLM (Summary + Judge)

In [5]:
from sm_sip.models import load_llm, create_summary_chain, create_judge_chain
from sm_sip.prompts import get_summary_prompt, get_judge_prompt

# Single LLM for both tasks
llm_model, llm_tokenizer, gen_pipe = load_llm(
    inference_config.llm_model_id,
    inference_config.quantization,
    seed=inference_config.seed,
)

# Two chains, same underlying model
summary_chain = create_summary_chain(gen_pipe, get_summary_prompt("it", "source_aware"))
judge_chain = create_judge_chain(gen_pipe, get_judge_prompt("unified"))

  Loading LLM (4bit, seed=42): meta-llama/Llama-3.1-8B-Instruct...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Step 3: Run Enhanced Evaluation

In [6]:
from sm_sip.pipelines import run_enhanced_evaluation

metrics, samples = run_enhanced_evaluation(
    processed_data,
    summary_chain,
    judge_chain=judge_chain,
    lang="it",
)

print("\n=== RESULTS ===")
for key, val in metrics.items():
    print(f"  {key}: {val['mean']:.4f} ± {val['std']:.4f}")

    Generating & Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedenc

  Computing BERTScore on CPU (batch)...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== RESULTS ===
  rouge1: 0.2033 ± 0.0984
  rougeL: 0.1255 ± 0.0528
  kir: 0.4397 ± 0.3153
  abstraction: 0.6086 ± 0.2027
  compression: 0.2590 ± 0.1491
  novel_ngrams: 0.4577 ± 0.1894
  judge_faithfulness: 4.9700 ± 0.1706
  judge_completeness: 4.9500 ± 0.2179
  judge_conciseness: 4.6600 ± 0.5517
  judge_abstraction: 4.9200 ± 0.4622
  bert: 0.6617 ± 0.0416


## Step 4: Save Results

In [7]:
from sm_sip.utils.io import save_results
from datetime import datetime

save_results({
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "sigext_model": sigext_config.model_id,
        "llm_model": inference_config.llm_model_id,
        "quantization": inference_config.quantization,
        "prompt_type": inference_config.prompt_type,
        "seed": inference_config.seed,
        "num_samples": len(samples),
    },
    "metrics": metrics,
    "samples": samples,
}, "results/italian/eval_enhanced.json")

print("Results saved!")

  Results saved: results/italian/eval_enhanced.json
Results saved!


## Cleanup

In [8]:
from sm_sip.utils.gpu import clear_gpu_memory
del llm_model, llm_tokenizer, gen_pipe
clear_gpu_memory()